# AI 预测蛋白质结合位点

在虚拟筛选流程中，**结合位点预测**是第一步：在对接之前，我们需要知道配体会结合到蛋白质的哪个位置。

## 范式概述

用图神经网络预测结合位点的范式可以概括为三步：

```
蛋白质 3D 结构 (PDB)           已知配体位置 (MOL2)
       │                              │
       ▼                              ▼
┌──────────────┐              ┌──────────────────┐
│ 1. 构建蛋白图 │              │ 生成距离软标签    │
│ 原子=节点     │              │ 离配体近→标签≈1   │
│ 空间近邻=边   │              │ 离配体远→标签≈0   │
└──────┬───────┘              └────────┬─────────┘
       │                              │
       ▼                              ▼
┌──────────────────────────────────────────────┐
│ 2. GNN 消息传递：每个原子聚合周围原子的信息     │
│    经过多轮传递后，每个原子「知道」周围的结构环境 │
└──────────────────┬───────────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────────┐
│ 3. 逐原子分类：每个原子输出一个概率             │
│    概率高 → 该原子属于结合位点                  │
│    概率低 → 该原子不属于结合位点                │
└──────────────────────────────────────────────┘
```

本教程以 GrASP 模型为例，完整演示这个范式。

## 学习目标

- 理解「蛋白质结构 → 图 → 逐原子分类」的范式
- 理解训练标签的生成方式（距离 → 软标签）
- 理解 GNN 如何通过消息传递捕获空间结构信息
- 学会评估结合位点预测模型（ROC AUC）

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import distance_matrix as scipy_dist_matrix
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import matplotlib.pyplot as plt

%matplotlib inline
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")

# 版本信息
display(pd.DataFrame({
    '库': ['torch', 'numpy', 'scipy', 'sklearn', 'matplotlib', 'pandas'],
    '版本': [torch.__version__, np.__version__,
            __import__('scipy').__version__, __import__('sklearn').__version__,
            __import__('matplotlib').__version__, pd.__version__]
}))

## 1. 超参数设置

两个最关键的超参数：

- **DISTANCE_CUTOFF = 5.0 Å**：两个蛋白原子距离小于 5 Å 就连一条边。这决定了图的稀疏程度。
- **LABEL_MIDPOINT = 5.0 Å**：蛋白原子到最近配体原子的距离小于 5 Å 时，标签接近 1（结合位点）；大于 5 Å 时，标签接近 0。

In [ ]:
# ---------- 超参数 ----------
DISTANCE_CUTOFF = 5.0       # 图的边距离阈值 (Angstrom)
ATOM_FEAT_DIM = 26          # 21(残基 one-hot) + 5(元素 one-hot)
HIDDEN_DIM = 64             # 隐层维度
N_GAT_LAYERS = 4            # 消息传递层数
N_HEADS = 4                 # 多头注意力头数
LABEL_MIDPOINT = 5.0        # 软标签 sigmoid 中点 (Angstrom)
LABEL_SLOPE = 3.0           # 软标签 sigmoid 斜率
NOISE_STD = 0.02            # 训练时添加的噪声强度
LOSS_WEIGHT_CLASS = 0.9     # 分类损失权重
LOSS_WEIGHT_RECON = 0.1     # 重建损失权重
N_EPOCHS = 150              # 训练轮数
LR = 5e-3                   # 学习率
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"计算设备: {DEVICE}")

## 2. 数据加载与特征提取

这一步是整个范式的基础：**把蛋白质 3D 结构变成图神经网络能处理的图数据**。

### 输入文件

| 文件 | 内容 | 用途 |
|------|------|------|
| `{pdbid}_pocket.pdb` | 蛋白质口袋的 3D 坐标 | 构建蛋白图的节点和边 |
| `{pdbid}_ligand.mol2` | 配体的 3D 坐标 | 生成训练标签（哪些原子离配体近） |

### 处理流程

1. **节点特征**：每个蛋白重原子编码为 26 维向量（残基类型 21 维 + 元素类型 5 维）
2. **边**：两个蛋白原子距离 < 5 Å 就连边，边权 = 1/距离
3. **标签**：每个蛋白原子到最近配体原子的距离，通过 sigmoid 函数转为 [0, 1] 的软标签

In [ ]:
# ---- 残基名称 -> one-hot 映射 (20 标准氨基酸 + 1 other = 21维) ----
RESIDUE_LIST = [
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
    'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'
]
RESIDUE_TO_IDX = {res: i for i, res in enumerate(RESIDUE_LIST)}

# ---- 元素符号 -> one-hot 映射 (C/N/O/S + other = 5维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S']
ELEMENT_TO_IDX = {e: i for i, e in enumerate(ELEMENT_LIST)}


def parse_pdb_ids(path):
    """从 CoreSet.dat 提取 PDB ID 列表。"""
    pdb_ids = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            pdb_ids.append(line.split()[0])
    return pdb_ids


def parse_pocket_pdb(filepath):
    """解析 pocket PDB 文件，提取重原子坐标和 26 维特征。"""
    coords = []
    feats = []
    with open(filepath, 'r') as f:
        for line in f:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue
            atom_name = line[12:16].strip()
            res_name = line[17:20].strip()
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            element = line[76:78].strip() if len(line) > 77 else atom_name[0]

            if element == 'H' or (len(element) == 0 and atom_name[0] == 'H'):
                continue

            feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
            if res_name in RESIDUE_TO_IDX:
                feat[RESIDUE_TO_IDX[res_name]] = 1.0
            else:
                feat[20] = 1.0
            if element in ELEMENT_TO_IDX:
                feat[21 + ELEMENT_TO_IDX[element]] = 1.0
            else:
                feat[25] = 1.0

            coords.append([x, y, z])
            feats.append(feat)

    if len(coords) == 0:
        return None, None
    return np.array(coords, dtype=np.float32), np.array(feats, dtype=np.float32)


def parse_ligand_mol2(filepath):
    """解析配体 MOL2 文件，提取重原子坐标（仅用于生成标签）。"""
    coords = []
    in_atom_section = False
    with open(filepath, 'r') as f:
        for line in f:
            line_stripped = line.strip()
            if line_stripped.startswith("@<TRIPOS>ATOM"):
                in_atom_section = True
                continue
            if line_stripped.startswith("@<TRIPOS>") and in_atom_section:
                break
            if not in_atom_section:
                continue
            parts = line_stripped.split()
            if len(parts) < 6:
                continue
            element = parts[5].split('.')[0]
            if element == 'H':
                continue
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            coords.append([x, y, z])

    if len(coords) == 0:
        return None
    return np.array(coords, dtype=np.float32)


def compute_soft_labels(prot_coords, lig_coords):
    """
    生成软标签：蛋白原子离配体越近，标签越接近 1。
    label = sigmoid(-slope * (d_min - midpoint))
    """
    dist_mat = scipy_dist_matrix(prot_coords, lig_coords)
    d_min = dist_mat.min(axis=1)
    x = -LABEL_SLOPE * (d_min - LABEL_MIDPOINT)
    labels = 1.0 / (1.0 + np.exp(-x))
    return labels.astype(np.float32)


def build_graph(coords, cutoff):
    """基于距离阈值构建空间邻近图。"""
    dist_mat = scipy_dist_matrix(coords, coords)
    src, dst = np.where((dist_mat < cutoff) & (dist_mat > 0))
    if len(src) == 0:
        nearest = np.argmin(dist_mat + np.eye(len(coords)) * 1e6, axis=1)
        src = np.arange(len(coords))
        dst = nearest
        src = np.concatenate([src, dst])
        dst_new = np.concatenate([nearest, np.arange(len(coords))])
        dst = dst_new

    edge_index = np.stack([src, dst], axis=0).astype(np.int64)
    edge_dist = dist_mat[src, dst]
    edge_weight = (1.0 / (edge_dist + 1e-6)).reshape(-1, 1).astype(np.float32)
    return edge_index, edge_weight


def build_complex_data(pdbid):
    """构建单个复合物的图数据 + 标签。"""
    pocket_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_pocket.pdb")
    ligand_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.mol2")

    if not os.path.exists(pocket_path) or not os.path.exists(ligand_path):
        return None

    prot_coords, prot_feats = parse_pocket_pdb(pocket_path)
    if prot_coords is None:
        return None

    lig_coords = parse_ligand_mol2(ligand_path)
    if lig_coords is None:
        return None

    labels = compute_soft_labels(prot_coords, lig_coords)
    edge_index, edge_weight = build_graph(prot_coords, DISTANCE_CUTOFF)

    return prot_feats, edge_index, edge_weight, labels

In [ ]:
# 展示一个样本，直观理解数据结构
pdb_ids = parse_pdb_ids(CORESET_FILE)
print(f"数据集包含 {len(pdb_ids)} 个蛋白-配体复合物\n")

sample_data = None
sample_id = None
for pid in sorted(pdb_ids):
    result = build_complex_data(pid)
    if result is not None:
        sample_data = result
        sample_id = pid
        break

if sample_data is not None:
    feats_s, ei_s, ew_s, labels_s = sample_data
    display(pd.DataFrame({
        '属性': ['PDB ID', '蛋白原子数（图的节点数）', '特征维度',
                '边数（空间近邻对）', '结合位点原子比例 (label > 0.5)'],
        '值': [sample_id, feats_s.shape[0], feats_s.shape[1],
              ei_s.shape[1], f"{(labels_s > 0.5).mean():.1%}"]
    }))
    print(f"\n标签分布：大部分原子远离配体（标签≈0），少数原子在结合位点附近（标签≈1）")

## 3. 数据集与数据加载器

将数据封装为 PyTorch Dataset。由于每个蛋白的原子数不同（变长图），逐样本处理。

In [ ]:
class BindingSiteDataset(Dataset):
    """结合位点预测数据集。每个样本 = 一个蛋白图 + 逐原子软标签。"""
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        feats, ei, ew, labels, pdbid = self.data[idx]
        return (torch.FloatTensor(feats),
                torch.LongTensor(ei),
                torch.FloatTensor(ew),
                torch.FloatTensor(labels),
                pdbid)


# ---- 预处理所有复合物 ----
print("正在构建数据...")
all_data = []
failed = 0
for pdbid in sorted(pdb_ids):
    result = build_complex_data(pdbid)
    if result is None:
        failed += 1
        continue
    feats, ei, ew, labels = result
    all_data.append((feats, ei, ew, labels, pdbid))

print(f"成功: {len(all_data)}, 失败: {failed}")

# ---- 80/20 划分 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_data = [all_data[i] for i in indices[:split]]
test_data = [all_data[i] for i in indices[split:]]

train_loader = DataLoader(BindingSiteDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(BindingSiteDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

display(pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    '复合物数': [len(train_data), len(test_data)]
}))

## 4. 模型架构

模型遵循经典的 **图神经网络逐节点分类** 范式：

```
原子特征 (N, 26)
    │
    ▼  编码器：线性变换到隐空间
隐层表示 (N, 64)
    │
    ▼  消息传递 × 4 轮：每个原子聚合周围原子的信息
更新后表示 (N, 64)
    │
    ▼  分类头：每个原子输出「是否为结合位点」的概率
预测 (N, 2)
```

消息传递是核心：一个原子本身的特征（残基类型、元素类型）不足以判断它是否是结合位点，
但如果它知道周围 5 Å 内都有哪些原子、形成了怎样的空间排布，判断就容易得多。
这正是 GNN 消息传递的作用——让每个原子「看到」周围的结构环境。

In [ ]:
def scatter_softmax(scores, index, num_nodes):
    """分组 softmax：对每个目标节点的所有入边分别做 softmax。"""
    max_scores = torch.full((num_nodes,), -1e9, device=scores.device)
    max_scores.scatter_reduce_(0, index, scores, reduce='amax')
    max_per_edge = max_scores[index]

    exp_scores = torch.exp(scores - max_per_edge)

    sum_exp = torch.zeros(num_nodes, device=scores.device)
    sum_exp.scatter_add_(0, index, exp_scores)
    sum_per_edge = sum_exp[index]

    return exp_scores / (sum_per_edge + 1e-8)


class GATv2Layer(nn.Module):
    """图注意力消息传递层：每个原子根据注意力权重聚合邻居信息。"""
    def __init__(self, in_dim, out_dim, n_heads, edge_dim=1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = out_dim // n_heads

        self.W_src = nn.Linear(in_dim, out_dim, bias=False)
        self.W_dst = nn.Linear(in_dim, out_dim, bias=False)
        self.W_edge = nn.Linear(edge_dim, out_dim, bias=False)
        self.W_val = nn.Linear(in_dim, out_dim, bias=False)

        self.attn_vec = nn.Parameter(torch.randn(n_heads, self.head_dim))
        nn.init.xavier_uniform_(self.attn_vec.unsqueeze(0))

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.layer_norm = nn.LayerNorm(out_dim)
        self.elu = nn.ELU()

    def forward(self, h, edge_index, edge_weight):
        src, dst = edge_index[0], edge_index[1]
        N = h.size(0)

        # 计算注意力分数
        combined = self.W_src(h[src]) + self.W_dst(h[dst]) + self.W_edge(edge_weight)
        combined = combined.view(-1, self.n_heads, self.head_dim)
        e = (self.leaky_relu(combined) * self.attn_vec).sum(dim=-1)

        # 归一化注意力
        alpha_heads = []
        for head in range(self.n_heads):
            alpha_heads.append(scatter_softmax(e[:, head], dst, N))
        alpha = torch.stack(alpha_heads, dim=-1)

        # 加权聚合邻居信息
        val = self.W_val(h[src]).view(-1, self.n_heads, self.head_dim)
        weighted_val = (alpha.unsqueeze(-1) * val).view(-1, self.n_heads * self.head_dim)

        out = torch.zeros(N, self.n_heads * self.head_dim, device=h.device)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(weighted_val), weighted_val)

        return self.elu(h + self.layer_norm(out))

In [ ]:
class GrASPToyModel(nn.Module):
    """
    结合位点预测模型：编码器 → 多轮消息传递 → 逐原子分类。
    附带一个重建辅助头用于正则化。
    """
    def __init__(self, in_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_layers=N_GAT_LAYERS, n_heads=N_HEADS):
        super().__init__()

        # 编码器
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.LayerNorm(hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.LayerNorm(hidden_dim), nn.ELU()
        )

        # 消息传递层
        self.gat_layers = nn.ModuleList([
            GATv2Layer(hidden_dim, hidden_dim, n_heads) for _ in range(n_layers)
        ])

        # 拼接所有层输出（Jumping Knowledge），融合多尺度信息
        jk_dim = hidden_dim * (n_layers + 1)

        # 分类头
        self.decoder = nn.Sequential(
            nn.Linear(jk_dim, 4 * hidden_dim), nn.ELU(),
            nn.Linear(4 * hidden_dim, 2 * hidden_dim), nn.ELU(),
            nn.Linear(2 * hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ELU(),
            nn.Linear(hidden_dim // 2, 2)
        )

        # 重建辅助头（正则化）
        self.recon_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, in_dim)
        )

    def forward(self, feats, edge_index, edge_weight):
        h = self.encoder(feats)

        # 收集每层输出用于 Jumping Knowledge
        jk_outputs = [h]
        for gat_layer in self.gat_layers:
            h = gat_layer(h, edge_index, edge_weight)
            jk_outputs.append(h)

        message_out = h
        jk = torch.cat(jk_outputs, dim=-1)

        pred = self.decoder(jk)
        recon = self.recon_head(message_out)
        return pred, recon


model = GrASPToyModel().to(DEVICE)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

## 5. 训练

训练目标：让模型学会从蛋白图结构中判断每个原子是否是结合位点。

两个训练技巧：
- **Noisy Nodes**：训练时对输入特征加少量噪声，防止过拟合
- **多任务损失**：分类损失 + 重建损失，后者要求模型保留原始特征信息

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)
ce_loss_fn = nn.CrossEntropyLoss()

print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for feats, ei, ew, labels, pdbid in train_loader:
        feats = feats.squeeze(0).to(DEVICE)
        ei = ei.squeeze(0).to(DEVICE)
        ew = ew.squeeze(0).to(DEVICE)
        labels = labels.squeeze(0).to(DEVICE)

        # Noisy Nodes: 训练时加噪声
        feat_std = feats.std(dim=0)
        noisy_feats = feats + feat_std * NOISE_STD * torch.randn_like(feats)

        pred, recon = model(noisy_feats, ei, ew)

        # 软标签转为二分类目标
        soft_2class = torch.stack([1.0 - labels, labels], dim=1)

        # 多任务损失
        loss_class = ce_loss_fn(pred, soft_2class)
        loss_recon = F.mse_loss(recon, feats)
        loss = LOSS_WEIGHT_CLASS * loss_class + LOSS_WEIGHT_RECON * loss_recon

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for feats, ei, ew, labels, pdbid in test_loader:
                feats = feats.squeeze(0).to(DEVICE)
                ei = ei.squeeze(0).to(DEVICE)
                ew = ew.squeeze(0).to(DEVICE)
                labels = labels.squeeze(0)

                pred, _ = model(feats, ei, ew)
                probs = pred.softmax(dim=-1)[:, 1].cpu().numpy()
                hard_labels = (labels.numpy() > 0.5).astype(int)
                all_labels.append(hard_labels)
                all_probs.append(probs)

        all_labels = np.concatenate(all_labels)
        all_probs = np.concatenate(all_probs)
        auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {np.mean(train_losses):.4f}  |  Test AUC: {auc:.4f}")

## 6. 评估与可视化

评估结合位点预测模型的核心指标是 **ROC AUC**：
- AUC = 1.0 表示完美区分结合位点和非结合位点原子
- AUC = 0.5 表示随机猜测

In [ ]:
model.eval()
all_labels, all_probs = [], []
per_complex_aucs = []

with torch.no_grad():
    for feats, ei, ew, labels, pdbid in test_loader:
        feats = feats.squeeze(0).to(DEVICE)
        ei = ei.squeeze(0).to(DEVICE)
        ew = ew.squeeze(0).to(DEVICE)
        labels = labels.squeeze(0)

        pred, _ = model(feats, ei, ew)
        probs = pred.softmax(dim=-1)[:, 1].cpu().numpy()
        hard_labels = (labels.numpy() > 0.5).astype(int)

        all_labels.append(hard_labels)
        all_probs.append(probs)

        if len(np.unique(hard_labels)) > 1:
            per_complex_aucs.append(roc_auc_score(hard_labels, probs))

all_labels = np.concatenate(all_labels)
all_probs = np.concatenate(all_probs)

global_auc = roc_auc_score(all_labels, all_probs)
global_ap = average_precision_score(all_labels, all_probs)
fpr, tpr, _ = roc_curve(all_labels, all_probs)

results = {'指标': ['Global ROC AUC', 'Global Avg Precision', '测试复合物数', '总原子数'],
           '值': [f"{global_auc:.4f}", f"{global_ap:.4f}", len(test_data), len(all_labels)]}
if per_complex_aucs:
    results['指标'].append('Per-complex AUC 均值')
    results['值'].append(f"{np.mean(per_complex_aucs):.4f}")
display(pd.DataFrame(results))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC 曲线
ax1 = axes[0]
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {global_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve', fontsize=13)
ax1.legend(loc='lower right', fontsize=11)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.02])

# Per-complex AUC 分布
ax2 = axes[1]
if per_complex_aucs:
    ax2.hist(per_complex_aucs, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax2.axvline(np.mean(per_complex_aucs), color='red', linestyle='--', lw=2,
                label=f'Mean = {np.mean(per_complex_aucs):.3f}')
    ax2.set_xlabel('Per-complex AUC', fontsize=12)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title('Per-complex AUC Distribution', fontsize=13)
    ax2.legend(loc='upper left', fontsize=11)
else:
    ax2.text(0.5, 0.5, 'Not enough data', ha='center', va='center',
             transform=ax2.transAxes, fontsize=14)

plt.tight_layout()
plt.show()

## 总结

本教程演示了用 GNN 预测蛋白质结合位点的完整范式：

1. **蛋白质 → 图**：原子作为节点，空间近邻关系作为边
2. **标签生成**：用蛋白原子到配体的距离，通过 sigmoid 转为软标签
3. **消息传递**：让每个原子「看到」周围的空间结构环境
4. **逐原子分类**：每个原子输出一个属于结合位点的概率

这个范式适用于多种结合位点预测模型，不同模型的区别主要在于消息传递层的设计和训练技巧，
但「构建图 → 消息传递 → 逐节点分类」的基本框架是通用的。